In [1]:
%cd /mnt/sdd1/atharvas/formulacode/datasmith/
import json
from pathlib import Path

import pandas as pd

from datasmith.docker.context import ContextRegistry, DockerContext, Task
from datasmith.notebooks.utils import update_cr

/mnt/sdd1/atharvas/formulacode/datasmith


02:46:54 WARNING  simple_useragent.core: Falling back to historic user agent.


In [2]:
verified_registry_pth = Path("/mnt/sdd1/atharvas/formulacode/datasmith/scratch/context_registry_final_filtered.json")
verified_repos_pth = Path(
    "/mnt/sdd1/atharvas/formulacode/terminal-bench/adapters/formulacode/example_task/aws_ecr_filtered_formulacode-verified.parquet"
)
verified_repos = pd.read_parquet(verified_repos_pth)

registry = update_cr(ContextRegistry.load_from_file(verified_registry_pth))

In [3]:
def get_task(repo_name, base_commit_sha):
    for task in registry.registry:
        repo_name = f"{task.owner}/{task.repo}"
        sha = task.sha
        if repo_name == repo_name and sha == base_commit_sha:
            return task, registry.registry[task]
    return None, None


new_registry = ContextRegistry()

verified_repos["is_available"] = False
for idx, row in verified_repos.iterrows():
    repo_name = row["repo_name"]
    base_commit_sha = row["pr_base"]["sha"]
    task, context = get_task(repo_name, base_commit_sha)
    if task is not None:
        verified_repos.at[idx, "is_available"] = True
        new_registry.register(task.with_tag("pkg"), context)


print(len(new_registry.registry))
new_registry.save_to_file(Path("scratch/formulacode_verified_context_registry.json"))

02:47:06 WARNING  datasmith.docker.context: Context 'Task(owner='pandas-dev', repo='pandas', sha='e1dd15b41e02ec78ca61379dad74a99f5ec16aa0', commit_date=1671302386.0, env_payload='{"dependencies": ["aiobotocore==2.25.1", "aiohappyeyeballs==2.6.1", "aiohttp==3.13.2", "aioitertools==0.12.0", "aiosignal==1.4.0", "attrs==25.4.0", "beautifulsoup4==4.14.2", "blosc2==3.11.0", "botocore==1.40.61", "bottleneck==1.6.0", "brotlipy==0.7.0", "cachetools==6.2.1", "certifi==2025.10.5", "cffi==2.0.0", "charset-normalizer==3.4.4", "contourpy==1.3.3", "cramjam==2.11.0", "cycler==0.12.1", "db-dtypes==1.4.3", "decorator==5.2.1", "defusedxml==0.7.1", "et-xmlfile==2.0.0", "execnet==2.1.1", "fastparquet==2024.11.0", "fonttools==4.60.1", "frozenlist==1.8.0", "fsspec==2025.10.0", "gcsfs==2025.10.0", "google-api-core==2.28.1", "google-auth==2.42.1", "google-auth-oauthlib==1.2.2", "google-cloud-bigquery==3.38.0", "google-cloud-core==2.5.0", "google-cloud-storage==3.4.1", "google-crc32c==1.7.1", "google-resumable

02:47:06 WARNING  datasmith.docker.context: Context 'Task(owner='networkx', repo='networkx', sha='f32dd409623285e0b67d0c07033b533414eb2bba', commit_date=1752070576.0, env_payload='{"dependencies": ["accessible-pygments==0.0.5", "affine==2.4.0", "alabaster==1.0.0", "asttokens==3.0.0", "attrs==25.3.0", "babel==2.17.0", "beautifulsoup4==4.13.4", "cairocffi==1.7.1", "certifi==2025.4.26", "cffi==1.17.1", "cfgv==3.4.0", "charset-normalizer==3.4.2", "click==8.2.1", "click-plugins==1.1.1", "cligj==0.7.2", "comm==0.2.2", "contextily==1.6.2", "contourpy==1.3.2", "coverage==7.8.2", "cycler==0.12.1", "debugpy==1.8.14", "decorator==5.2.1", "distlib==0.3.9", "docutils==0.21.2", "execnet==2.1.1", "executing==2.2.0", "fastjsonschema==2.21.1", "filelock==3.18.0", "fonttools==4.58.2", "geographiclib==2.0", "geopandas==1.1.0", "geopy==2.4.1", "greenlet==3.2.3", "identify==2.6.12", "idna==3.10", "igraph==0.11.8", "imagesize==1.4.1", "importlib-metadata==8.7.0", "iniconfig==2.1.0", "intersphinx-registry==0

187


In [6]:
# save each dockerfile context in a folder called formulacode_verified/{repo_name}/{sha}/{all files}
def save_context(context: DockerContext, task: Task, task_dir: Path):
    task_dir.mkdir(parents=True, exist_ok=True)
    task_dir.joinpath("Dockerfile").write_text(context.dockerfile_data)
    task_dir.joinpath("entrypoint.sh").write_text(context.entrypoint_data)
    task_dir.joinpath("docker_build_base.sh").write_text(context.base_building_data)
    task_dir.joinpath("docker_build_run.sh").write_text(context.run_building_data)
    task_dir.joinpath("docker_build_env.sh").write_text(context.env_building_data)
    task_dir.joinpath("docker_build_final.sh").write_text(context.final_building_data)
    task_dir.joinpath("docker_build_pkg.sh").write_text(context.building_data)
    task_dir.joinpath("profile.sh").write_text(context.profile_data)
    task_dir.joinpath("run_tests.sh").write_text(context.run_tests_data)
    task_dir.joinpath("task.txt").write_text(repr(task))


for _, row in verified_repos.iterrows():
    task_id = row["task_id"]
    repo_name = row["repo_name"]
    base_commit_sha = row["pr_base"]["sha"]
    task, context = get_task(repo_name, base_commit_sha)
    if not task or not context:
        print(f"Skipping {task_id} as context not found")
        continue
    task_dir = Path("dataset/formulacode_verified") / repo_name.replace("/", "_") / base_commit_sha
    save_context(context, task, task_dir)

In [7]:
Path("dataset/config.json").write_text(
    json.dumps({
        "context_registry_path": "scratch/formulacode_verified_context_registry.json",
        "ecr_repo": "formulacode/all",
        "aws_region": "us-east-1",
    })
)

137